# Module 4 — Exploratory Data Analysis
Reuses `03_Cleaned_Data` + `src/eda_utils.py`.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append(str(Path.cwd() / "src"))
from eda_utils import (
    REPORT_DIR, CHART_DIR, SPENDING_COLS, PURCHASE_COLS, CAMPAIGN_COLS,
    load_cleaned_data, save_fig, save_excel,
    top_correlations, add_engineered_columns, add_segments,
)

pd.set_option("display.max_columns", None)
sns.set_style("whitegrid")


In [ ]:
df = load_cleaned_data()
df = add_engineered_columns(df)
df.head()


## 1. Dataset Overview

In [ ]:
shape = df.shape
dtypes = df.dtypes.astype(str)
missing = df.isnull().sum()
duplicates = df.duplicated().sum()

print(shape)
df.info()


In [ ]:
df.describe()

In [ ]:
dataset_summary = pd.DataFrame({
    "Column": df.columns,
    "Dtype": dtypes.reindex(df.columns).values,
    "Missing": missing.reindex(df.columns).values,
})
dataset_summary.loc["__meta__"] = ["Rows", shape[0], "Duplicates: " + str(duplicates)]
save_excel(dataset_summary, "dataset_summary")
dataset_summary


## 2. Demographic Analysis

In [ ]:
fig, ax = plt.subplots()
ax.hist(df["Age"], bins=20, color="steelblue")
ax.set_title("Age Distribution")
save_fig(fig, "age_histogram")

fig, ax = plt.subplots()
ax.boxplot(df["Income"])
ax.set_title("Income Boxplot")
save_fig(fig, "income_boxplot")

fig, ax = plt.subplots()
sns.countplot(data=df, x="Education", ax=ax, order=df["Education"].value_counts().index)
ax.set_title("Education Countplot")
save_fig(fig, "education_countplot")

fig, ax = plt.subplots()
df["Marital_Status"].value_counts().plot.pie(autopct="%1.1f%%", ax=ax)
ax.set_ylabel("")
ax.set_title("Marital Status Share")
save_fig(fig, "marital_status_pie")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.countplot(data=df, x="Kidhome", ax=axes[0])
sns.countplot(data=df, x="Teenhome", ax=axes[1])
axes[0].set_title("Kidhome")
axes[1].set_title("Teenhome")
save_fig(fig, "kidhome_teenhome_countplot")


In [ ]:
demographic_summary = pd.DataFrame({
    "Metric": ["Age Mean", "Age Median", "Income Mean", "Income Median",
               "Kidhome Mean", "Teenhome Mean"],
    "Value": [df["Age"].mean(), df["Age"].median(), df["Income"].mean(),
              df["Income"].median(), df["Kidhome"].mean(), df["Teenhome"].mean()],
})
save_excel(demographic_summary, "demographic_summary")
demographic_summary


## 3. Spending Analysis
`Total_Spending` = sum of all `Mnt*` columns.

In [ ]:
category_spending = df[SPENDING_COLS].sum().sort_values(ascending=False)
avg_spending = df["Total_Spending"].mean()
top_spender = df.loc[df["Total_Spending"].idxmax(), ["ID", "Total_Spending"]]

print("Average Total_Spending:", round(avg_spending, 2))
print("Top spender:\n", top_spender)
category_spending


In [ ]:
fig, ax = plt.subplots()
category_spending.plot.bar(ax=ax)
ax.set_title("Total Spending by Category")
save_fig(fig, "spending_by_category_bar")

fig, ax = plt.subplots()
ax.hist(df["Total_Spending"], bins=30, color="darkorange")
ax.set_title("Total Spending Distribution")
save_fig(fig, "total_spending_histogram")

fig, ax = plt.subplots()
ax.boxplot(df["Total_Spending"])
ax.set_title("Total Spending Boxplot")
save_fig(fig, "total_spending_boxplot")

edu_spending = df.groupby("Education")[SPENDING_COLS].sum()
fig, ax = plt.subplots()
edu_spending.plot(kind="bar", stacked=True, ax=ax)
ax.set_title("Spending by Category per Education (Stacked)")
save_fig(fig, "spending_stacked_bar")


In [ ]:
spending_summary = category_spending.reset_index()
spending_summary.columns = ["Category", "Total_Spending"]
spending_summary.loc[len(spending_summary)] = ["Average_Total_Spending", avg_spending]
save_excel(spending_summary, "spending_summary")
spending_summary


## 4. Purchasing Behaviour
`Total_Purchases` = sum of Web/Store/Catalog/Deals purchases.

In [ ]:
channel_totals = df[PURCHASE_COLS].sum().sort_values(ascending=False)
channel_totals


In [ ]:
fig, ax = plt.subplots()
channel_totals.plot.bar(ax=ax)
ax.set_title("Purchases by Channel")
save_fig(fig, "purchases_by_channel_bar")

fig, ax = plt.subplots()
ax.hist(df["Total_Purchases"], bins=20, color="seagreen")
ax.set_title("Total Purchases Distribution")
save_fig(fig, "total_purchases_histogram")

fig, ax = plt.subplots()
sns.countplot(data=df, x="NumDealsPurchases", ax=ax)
ax.set_title("Deal Purchases Countplot")
save_fig(fig, "deals_countplot")


In [ ]:
purchase_summary = channel_totals.reset_index()
purchase_summary.columns = ["Channel", "Total"]
purchase_summary.loc[len(purchase_summary)] = ["Avg_Total_Purchases", df["Total_Purchases"].mean()]
save_excel(purchase_summary, "purchase_summary")
purchase_summary


## 5. Website Engagement

In [ ]:
engagement_median = df["NumWebVisitsMonth"].median()
high_engagement = (df["NumWebVisitsMonth"] > engagement_median).sum()
low_engagement = (df["NumWebVisitsMonth"] <= engagement_median).sum()

fig, ax = plt.subplots()
ax.hist(df["NumWebVisitsMonth"], bins=15, color="slateblue")
ax.set_title("Web Visits per Month")
save_fig(fig, "webvisits_histogram")

fig, ax = plt.subplots()
ax.scatter(df["NumWebVisitsMonth"], df["Total_Spending"], alpha=0.4)
ax.set_xlabel("NumWebVisitsMonth")
ax.set_ylabel("Total_Spending")
ax.set_title("Web Visits vs Spending")
save_fig(fig, "webvisits_scatter")

fig, ax = plt.subplots()
sns.kdeplot(df["NumWebVisitsMonth"], ax=ax, fill=True)
ax.set_title("Web Visits KDE")
save_fig(fig, "webvisits_kde")


In [ ]:
website_summary = pd.DataFrame({
    "Metric": ["Median Visits", "High Engagement Count", "Low Engagement Count"],
    "Value": [engagement_median, high_engagement, low_engagement],
})
save_excel(website_summary, "website_summary")
website_summary


## 6. Recency Analysis

In [ ]:
recency_threshold = 30
active_customers = (df["Recency"] <= recency_threshold).sum()
inactive_customers = (df["Recency"] > recency_threshold).sum()

fig, ax = plt.subplots()
ax.hist(df["Recency"], bins=20, color="teal")
ax.set_title("Recency Distribution")
save_fig(fig, "recency_histogram")

print(f"Active (<= {recency_threshold} days): {active_customers}")
print(f"Inactive (> {recency_threshold} days): {inactive_customers}")
print("Recommendation: target inactive customers with re-engagement offers/discounts.")


## 7. Campaign Analysis

In [ ]:
campaign_all_cols = CAMPAIGN_COLS + ["Response"]
acceptance_rates = df[campaign_all_cols].mean().sort_values(ascending=False) * 100

fig, ax = plt.subplots()
acceptance_rates.plot.bar(ax=ax)
ax.set_ylabel("Acceptance Rate (%)")
ax.set_title("Campaign Acceptance Rates")
save_fig(fig, "campaign_acceptance_bar")

fig, ax = plt.subplots()
sns.countplot(data=df, x="Response", ax=ax)
ax.set_title("Response Countplot")
save_fig(fig, "response_countplot")

fig, ax = plt.subplots()
df["Response"].value_counts().plot.pie(autopct="%1.1f%%", ax=ax)
ax.set_ylabel("")
ax.set_title("Response Share")
save_fig(fig, "response_pie")


In [ ]:
campaign_summary = acceptance_rates.reset_index()
campaign_summary.columns = ["Campaign", "Acceptance_Rate_%"]
save_excel(campaign_summary, "campaign_summary")
campaign_summary


## 8. Complaint Analysis

In [ ]:
complaint_pct = df["Complain"].mean() * 100
complaint_vs_spending = df.groupby("Complain")["Total_Spending"].mean()
complaint_vs_income = df.groupby("Complain")["Income"].mean()

fig, ax = plt.subplots()
sns.countplot(data=df, x="Complain", ax=ax)
ax.set_title("Complaint Countplot")
save_fig(fig, "complaint_countplot")

heat_data = df.groupby("Complain")[["Total_Spending", "Income"]].mean()
fig, ax = plt.subplots()
sns.heatmap(heat_data.T, annot=True, fmt=".0f", cmap="coolwarm", ax=ax)
ax.set_title("Complaint vs Spending/Income")
save_fig(fig, "complaint_heatmap")


In [ ]:
complaint_summary = pd.DataFrame({
    "Complain": complaint_vs_spending.index,
    "Avg_Spending": complaint_vs_spending.values,
    "Avg_Income": complaint_vs_income.values,
})
complaint_summary.loc[len(complaint_summary)] = ["Complaint_%", complaint_pct, np.nan]
save_excel(complaint_summary, "complaint_summary")
complaint_summary


## 9. Correlation Analysis

In [ ]:
numeric_df = df.select_dtypes(include=np.number).drop(columns=["ID"])
corr = numeric_df.corr(method="pearson")

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(corr, cmap="coolwarm", center=0, ax=ax)
ax.set_title("Correlation Matrix (Pearson)")
save_fig(fig, "correlation_heatmap")

top_pos, top_neg = top_correlations(corr, n=5)
print(top_pos)
print(top_neg)


In [ ]:
correlation_summary = pd.concat([
    top_pos.rename("Correlation").reset_index().assign(Type="Top Positive"),
    top_neg.rename("Correlation").reset_index().assign(Type="Top Negative"),
], ignore_index=True)
correlation_summary.columns = ["Var1", "Var2", "Correlation", "Type"]
save_excel(correlation_summary, "correlation_summary")
correlation_summary


## 10. Customer Segmentation

In [ ]:
segments = add_segments(df)
segment_counts = segments.sum().rename("Count").reset_index()
segment_counts.columns = ["Segment", "Count"]
segment_counts["Percent"] = (segment_counts["Count"] / len(df) * 100).round(2)
save_excel(segment_counts, "customer_segments")
segment_counts


## 11. Business Insights

In [ ]:
best_channel = channel_totals.idxmax()
best_campaign = acceptance_rates.idxmax()
income_spending_corr = df["Income"].corr(df["Total_Spending"])
children_spending_corr = (df["Kidhome"] + df["Teenhome"]).corr(df["Total_Spending"])
high_value_profile = df.loc[segments["High_Value"], ["Age", "Income", "Total_Spending"]].mean()

insights = pd.DataFrame({
    "Insight": [
        "Highest spending customer (ID)",
        "Highest revenue category",
        "Income vs Spending correlation",
        "Most popular purchase channel",
        "Best performing campaign",
        "Most responsive segment",
        "Inactive customers (count)",
        "High engagement customers (count)",
        "Children vs Spending correlation",
        "High-Value avg Age",
        "High-Value avg Income",
        "High-Value avg Spending",
        "Strongest positive correlation",
        "Strongest negative correlation",
        "Marketing recommendation",
    ],
    "Value": [
        int(top_spender["ID"]),
        category_spending.idxmax(),
        round(income_spending_corr, 3),
        best_channel,
        best_campaign,
        "Campaign_Responder segment (see customer_segments.xlsx)",
        int(inactive_customers),
        int(high_engagement),
        round(children_spending_corr, 3),
        round(high_value_profile["Age"], 1),
        round(high_value_profile["Income"], 2),
        round(high_value_profile["Total_Spending"], 2),
        f"{top_pos.index[0]} ({top_pos.iloc[0]:.2f})",
        f"{top_neg.index[0]} ({top_neg.iloc[0]:.2f})",
        "Prioritize wine/meat buyers, re-target inactive customers, "
        "and expand the best-performing campaign to high-value segments.",
    ],
})
save_excel(insights, "business_insights")
insights
